# Healthcare Operations SQL Validation Notebook

This notebook supports the validation expectations for AI-assisted SQL work. Use it before accepting an AI-generated query as correct. It helps you check schema fit, row counts, date ranges, status values, joins, filters, and expected outputs.

## Validation workflow

Use these cells in order when reviewing AI-generated SQL:

1. Confirm the database connection and available tables.
2. Inspect schemas and row counts.
3. Check fixed reporting dates and valid status values.
4. Check joins before trusting multi-table results.
5. Add filters one at a time.
6. Run expected-output checks or sanity checks.

In [1]:
import sqlite3
import pandas as pd

DB_FILE = "healthcare_operations.db"
conn = sqlite3.connect(DB_FILE)
print(f"Connected to {DB_FILE}")

Connected to healthcare_operations.db


## 1. Table list check

Start by confirming that the database contains the tables your query expects. This helps catch prompts or SQL that assume the wrong database setup.

In [2]:
tables = pd.read_sql("""
SELECT name AS table_name
FROM sqlite_master
WHERE type = 'table'
  AND name NOT LIKE 'sqlite_%'
ORDER BY name;
""", conn)
tables

,table_name
0,appointments
1,clinics
2,patients
3,providers
4,reporting_context


## 2. Schema inspection

Use the schema as the source of truth. Any table or column that does not appear here should be treated as a possible hallucination.

In [3]:
for table_name in tables['table_name']:
    print(f"\n--- {table_name} ---")
    display(pd.read_sql(f"PRAGMA table_info({table_name});", conn))


--- appointments ---


,cid,name,type,notnull,dflt_value,pk
0,0,appointment_id,INTEGER,0,None,1
1,1,patient_id,INTEGER,1,None,0
2,2,provider_id,INTEGER,1,None,0
3,3,clinic_id,INTEGER,1,None,0
4,4,appointment_date,TEXT,1,None,0
5,5,appointment_status,TEXT,1,None,0
6,6,appointment_type,TEXT,1,None,0



--- clinics ---


,cid,name,type,notnull,dflt_value,pk
0,0,clinic_id,INTEGER,0,None,1
1,1,clinic_name,TEXT,1,None,0
2,2,city,TEXT,1,None,0
3,3,region,TEXT,1,None,0



--- patients ---


,cid,name,type,notnull,dflt_value,pk
0,0,patient_id,INTEGER,0,None,1
1,1,patient_name,TEXT,0,None,0
2,2,date_of_birth,TEXT,0,None,0
3,3,city,TEXT,0,None,0
4,4,insurance_type,TEXT,0,None,0



--- providers ---


,cid,name,type,notnull,dflt_value,pk
0,0,provider_id,INTEGER,0,None,1
1,1,provider_name,TEXT,0,None,0
2,2,specialty,TEXT,0,None,0



--- reporting_context ---


,cid,name,type,notnull,dflt_value,pk
0,0,context_id,INTEGER,0,None,1
1,1,reporting_date,TEXT,1,None,0
2,2,data_start_date,TEXT,1,None,0
3,3,data_end_date,TEXT,1,None,0
4,4,recent_30_day_start,TEXT,1,None,0
5,5,recent_60_day_start,TEXT,1,None,0
6,6,recent_3_month_start,TEXT,1,None,0
7,7,recent_6_month_start,TEXT,1,None,0
8,8,primary_date_column,TEXT,1,None,0
9,9,note,TEXT,0,None,0


## 3. Row counts

Check row counts before and after writing complex queries. If a source table has zero rows, a query that returns no rows may be behaving correctly.

In [4]:
row_counts = []
for table_name in tables['table_name']:
    count = pd.read_sql(f"SELECT COUNT(*) AS row_count FROM {table_name};", conn).iloc[0, 0]
    row_counts.append({'table_name': table_name, 'row_count': count})
pd.DataFrame(row_counts)

,table_name,row_count
0,appointments,120
1,clinics,3
2,patients,50
3,providers,10
4,reporting_context,1


## 4. Reporting context and min/max dates

Use the fixed reporting context instead of live-date logic such as `DATE('now')`. This prevents fictional historical datasets from returning empty or misleading results.

In [5]:
pd.read_sql("SELECT * FROM reporting_context;", conn)

,context_id,reporting_date,data_start_date,data_end_date,recent_30_day_start,recent_60_day_start,recent_3_month_start,recent_6_month_start,primary_date_column,note
0,1,2025-06-29,2025-01-06,2025-06-29,2025-05-30,2025-04-30,2025-03-29,2024-12-29,appointments.appointment_date,Use reporting_date instead of date('now') beca...


In [6]:
pd.read_sql("SELECT * FROM dataset_date_range;", conn)

,table_name,date_column,min_date,max_date,row_count
0,appointments,appointment_date,2025-01-06,2025-06-29,120


## 5. Valid status and category values

Check exact spelling and capitalization before writing filters. Small mismatches such as `no-show` vs. `no_show` or `canceled` vs. `Canceled` can change results.

In [7]:
pd.read_sql("SELECT * FROM status_value_counts ORDER BY table_name, status_column, row_count DESC;", conn)

,table_name,status_column,status_value,row_count
0,appointments,appointment_status,completed,87
1,appointments,appointment_status,no_show,23
2,appointments,appointment_status,canceled,10
3,appointments,appointment_type,follow_up,36
4,appointments,appointment_type,consult,32
5,appointments,appointment_type,checkup,29
6,appointments,appointment_type,screening,23


## 6. Join coverage checks

Check whether appointment records match valid patients, providers, and clinics before trusting a multi-table query.

In [8]:
pd.read_sql("""
SELECT
    COUNT(*) AS total_appointments,
    COUNT(p.patient_id) AS appointments_with_matching_patient,
    COUNT(pr.provider_id) AS appointments_with_matching_provider,
    COUNT(c.clinic_id) AS appointments_with_matching_clinic
FROM appointments AS a
LEFT JOIN patients AS p
    ON a.patient_id = p.patient_id
LEFT JOIN providers AS pr
    ON a.provider_id = pr.provider_id
LEFT JOIN clinics AS c
    ON a.clinic_id = c.clinic_id;
""", conn)

,total_appointments,appointments_with_matching_patient,appointments_with_matching_provider,appointments_with_matching_clinic
0,120,120,120,120


## 7. Row-count-before-and-after join checks

These joins should preserve appointment-level rows when each appointment links to one patient, provider, and clinic.

In [9]:
pd.read_sql("""
SELECT 'appointments only' AS query_step, COUNT(*) AS row_count FROM appointments
UNION ALL
SELECT 'appointments + patients', COUNT(*)
FROM appointments AS a
JOIN patients AS p
    ON a.patient_id = p.patient_id
UNION ALL
SELECT 'appointments + patients + providers + clinics', COUNT(*)
FROM appointments AS a
JOIN patients AS p
    ON a.patient_id = p.patient_id
JOIN providers AS pr
    ON a.provider_id = pr.provider_id
JOIN clinics AS c
    ON a.clinic_id = c.clinic_id;
""", conn)

,query_step,row_count
0,appointments only,120
1,appointments + patients,120
2,appointments + patients + providers + clinics,120


## 8. Filter-by-filter debugging

Use exact status values from the database. In this dataset, missed appointments are stored as `no_show`.

In [10]:
pd.read_sql("""
SELECT 'all appointments' AS step, COUNT(*) AS row_count FROM appointments
UNION ALL
SELECT 'last 3 months', COUNT(*)
FROM appointments
WHERE appointment_date >= (SELECT recent_3_month_start FROM reporting_context)
UNION ALL
SELECT 'last 3 months and no_show', COUNT(*)
FROM appointments
WHERE appointment_date >= (SELECT recent_3_month_start FROM reporting_context)
  AND appointment_status = 'no_show';
""", conn)

,step,row_count
0,all appointments,120
1,last 3 months,58
2,last 3 months and no_show,15


## 9. Expected-output check: repeat no-show patients

This validates a common healthcare lab pattern: patients with repeated no-shows in the anchored recent period.

In [11]:
repeat_no_show_patients = pd.read_sql("""
SELECT
    p.patient_id,
    p.patient_name,
    COUNT(a.appointment_id) AS no_show_count
FROM patients AS p
JOIN appointments AS a
    ON p.patient_id = a.patient_id
WHERE a.appointment_date >= (
    SELECT recent_3_month_start
    FROM reporting_context
)
  AND a.appointment_status = 'no_show'
GROUP BY p.patient_id, p.patient_name
HAVING COUNT(a.appointment_id) >= 2
ORDER BY no_show_count DESC, p.patient_name;
""", conn)

expected_columns = ['patient_id', 'patient_name', 'no_show_count']
assert list(repeat_no_show_patients.columns) == expected_columns
repeat_no_show_patients

,patient_id,patient_name,no_show_count
0,28,Brandon Carter,2
1,39,Maya Stewart,2
2,17,Quinn Lewis,2
3,46,Thomas Bell,2


In [12]:
# Context check: total scheduled vs missed per patient in the recent window
pd.read_sql("""
SELECT
    a.patient_id,
    COUNT(*) AS total_scheduled,
    SUM(CASE WHEN a.appointment_status IN ('no_show', 'canceled') THEN 1 ELSE 0 END) AS total_missed
FROM appointments AS a
WHERE a.appointment_date BETWEEN (SELECT recent_3_month_start FROM reporting_context)
                             AND (SELECT reporting_date FROM reporting_context)
GROUP BY a.patient_id
HAVING total_missed >= 1
ORDER BY total_missed DESC, total_scheduled;
""", conn)

,patient_id,total_scheduled,total_missed
0,28,4,3
1,17,2,2
2,39,3,2
3,46,3,2
4,5,1,1
5,10,1,1
6,14,1,1
7,20,1,1
8,23,1,1
9,25,1,1


# AI-Assisted SQL Workflow: Validation and Refinement

## Task 1: Revisit and Clarify the Problem

**Original business question (previous lab):** Which patients had two or more missed appointments (no-shows or cancellations) in the most recent 3-month reporting period?

**Original key definitions:**
- **Missed appointment:** `appointment_status IN ('no_show', 'canceled')`
- **At-risk patient:** 2 or more missed appointments in the recent period
- **Recent period:** `reporting_context.recent_3_month_start` (2025-03-29) through `reporting_context.reporting_date` (2025-06-29)

**Why it needs clarifying:** The original output ranked patients by raw missed count only. The context check above shows this misleads. Quinn Lewis missed **2 of 2** appointments (100%) but ranked 3rd, behind patients who attended some of theirs. A count without a denominator can't tell outreach staff who is most disengaged.

**Updated business question:** Among patients with 2 or more missed appointments in the recent 3-month period, what share of their scheduled appointments did they miss, and how recently? Rank them so clinics can prioritize outreach to the least engaged patients first.

**Updated definitions:**
- **Missed appointment:** unchanged, but no-shows and cancellations are reported **separately**, because the data does not record who canceled.
- **Total scheduled:** *all* appointments in the window (any status), the denominator for the miss rate.
- **Miss rate:** `total_missed / total_scheduled`, shown as a percentage with 1 decimal.
- **At-risk patient:** unchanged (2 or more missed), which keeps the minimum sample size per patient.

**Expected output:** one row per at-risk patient with `patient_id`, `patient_name`, `insurance_type`, `total_scheduled`, `no_show_count`, `canceled_count`, `total_missed`, `miss_rate_pct`, and `last_missed_date`, sorted by `miss_rate_pct` DESC, then `total_missed` DESC, then `patient_name`.

**What "correct and reliable" means for this analysis:**
1. **Runs in SQLite** with no errors.
2. **Schema-valid:** every table, column, and join exists in the database.
3. **Exact definitions:** the status values match the database exactly, and the window is anchored to `reporting_context` (inclusive on both ends).
4. **Consistent denominator:** `total_scheduled` counts all statuses in the *same* window as the missed counts.
5. **Reconciles:** `no_show_count + canceled_count = total_missed`, and `total_missed <= total_scheduled`.
6. **Independently verified:** a separate pandas calculation returns identical results.
7. **Decision-ready:** the ranking reflects actual engagement, not tie-break order, and the limitations are stated.

## Task 2: Validation Criteria and Checklist

Each check has a pass condition and a method. This checklist is applied to the original SQL in Task 3 and to the revised SQL in Task 7.

| # | Category | Check | How it is verified |
|---|---|---|---|
| 1 | Schema | Only real tables are used (`patients`, `appointments`, `reporting_context`) | Compare against the table list (validation cell 4) |
| 2 | Schema | Every column exists. No hallucinated fields. | Compare against the `PRAGMA table_info` output (cell 6) |
| 3 | Joins | Join key is `patients.patient_id = appointments.patient_id` | Relationship reference; join coverage (cells 15, 17) |
| 4 | Joins | The join neither drops nor duplicates appointment rows | Row count before and after the join stays at 120 (cell 17) |
| 5 | Filters | Status values match the database exactly (`'no_show'`, `'canceled'`) | `status_value_counts` (cell 13) |
| 6 | Filters | The window is anchored to `reporting_context`, inclusive on both ends, with no `date('now')` | Read the SQL; compare against `reporting_context` (cell 10) |
| 7 | Grouping | One row per patient. All non-aggregated columns are in `GROUP BY`. | Check that `patient_id` is unique in the output |
| 8 | Calculations | `no_show_count + canceled_count = total_missed` | Assertion on every row |
| 9 | Calculations | The denominator (`total_scheduled`) uses the same window and includes all statuses | Read the SQL; assert `total_missed <= total_scheduled` |
| 10 | Calculations | The miss rate uses decimal (not integer) division | Check for a value like 66.7, not 0 |
| 11 | Business alignment | The threshold of 2 or more missed is applied to the right count | Assert `total_missed >= 2` |
| 12 | Business alignment | The ranking reflects engagement (rate first), not tie-break order | Inspect the sort; compare with the context check |
| 13 | Completeness | Every planned output column is present, named, and in order | Assert the column list |
| 14 | Hallucinations and assumptions | No invented tables, columns, statuses, or SQL dialect features | Checks 1, 2, 5; runs in SQLite |
| 15 | Edge cases | Patients with no appointments, single-appointment patients, and ties are handled sensibly | Inspect the output; review the threshold logic |
| 16 | Independent verification | Results match a separate pandas recomputation from the raw rows | Assertion comparing both results |

**Edge cases considered:**
- **Integer division:** in SQLite, `2 / 3` evaluates to `0`, so the miss rate must use `100.0 *` or `CAST(... AS REAL)`.
- **Boundary dates:** appointments exactly on 2025-03-29 or 2025-06-29 must be included.
- **Single missed appointment (1 of 1):** that's a 100% miss rate but excluded by the 2-missed threshold. This is intentional, since one event is too little evidence to label a patient at risk.
- **Ties:** patients with equal rates are broken by `total_missed`, then name, so the order is deterministic.

In [13]:
# Task 3: rerun the original (previous lab's final) AI-generated SQL exactly as accepted
original_sql = """
SELECT
  p.patient_id,
  p.patient_name,
  p.insurance_type,
  SUM(CASE WHEN a.appointment_status = 'no_show' THEN 1 ELSE 0 END) AS no_show_count,
  SUM(CASE WHEN a.appointment_status = 'canceled' THEN 1 ELSE 0 END) AS canceled_count,
  COUNT(*) AS total_missed
FROM patients p
JOIN appointments a ON p.patient_id = a.patient_id
WHERE a.appointment_date >= (SELECT recent_3_month_start FROM reporting_context)
  AND a.appointment_date <= (SELECT reporting_date FROM reporting_context)
  AND a.appointment_status IN ('no_show', 'canceled')
GROUP BY p.patient_id, p.patient_name, p.insurance_type
HAVING COUNT(*) >= 2
ORDER BY total_missed DESC, p.patient_name ASC;
"""
original_result = pd.read_sql(original_sql, conn)

# Apply the checklist items that can be tested on this output
planned_columns = ['patient_id', 'patient_name', 'insurance_type', 'total_scheduled',
                   'no_show_count', 'canceled_count', 'total_missed',
                   'miss_rate_pct', 'last_missed_date']
checks = {
    "#7  One row per patient": original_result['patient_id'].is_unique,
    "#8  no_show + canceled = total_missed": (original_result['no_show_count']
                                              + original_result['canceled_count']
                                              == original_result['total_missed']).all(),
    "#11 Threshold total_missed >= 2": (original_result['total_missed'] >= 2).all(),
    "#13 All planned columns present": set(planned_columns) <= set(original_result.columns),
}
for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {name}")
print("Missing columns:", [c for c in planned_columns if c not in original_result.columns])

original_result

PASS  #7  One row per patient
PASS  #8  no_show + canceled = total_missed
PASS  #11 Threshold total_missed >= 2
FAIL  #13 All planned columns present
Missing columns: ['total_scheduled', 'miss_rate_pct', 'last_missed_date']


,patient_id,patient_name,insurance_type,no_show_count,canceled_count,total_missed
0,28,Brandon Carter,HMO,2,1,3
1,39,Maya Stewart,Medicaid,2,0,2
2,17,Quinn Lewis,Medicaid,2,0,2
3,46,Thomas Bell,Medicare,2,0,2


## Task 3: Review of the Existing SQL Output

The query above is the previous lab's final AI-generated SQL. It passed every check in that lab. Here it is reviewed against the Task 2 checklist and the updated business question.

| # | Check | Result | Notes |
|---|---|---|---|
| 1–2 | Real tables and columns | ✅ Pass | Only `patients`, `appointments`, `reporting_context`. All columns exist. |
| 3–4 | Join key and row preservation | ✅ Pass | Correct key. Cell 17 confirms that all 120 appointments match a patient. |
| 5 | Exact status values | ✅ Pass | `'no_show'` and `'canceled'` match the database exactly. |
| 6 | Anchored, inclusive window | ✅ Pass | Uses `reporting_context` with `>=` and `<=`. |
| 7 | One row per patient | ✅ Pass | `patient_id` is unique. |
| 8 | Counts reconcile | ✅ Pass | `no_show_count + canceled_count = total_missed` on every row. |
| 9 | Denominator present | ❌ Fail | There is no `total_scheduled`. The `WHERE` clause keeps **only** missed rows, so completed appointments are removed before grouping and the query **structurally cannot** compute a denominator. |
| 10 | Rate calculation | ❌ Fail | No miss rate exists. |
| 11 | Threshold applied | ✅ Pass | `HAVING COUNT(*) >= 2`. |
| 12 | Ranking reflects engagement | ❌ Fail | It ranks by raw count, then **alphabetically**. Quinn Lewis (2 of 2 missed, 100%) ranks 3rd, behind Maya Stewart (2 of 3, 67%), only because "M" comes before "Q". |
| 13 | All planned columns | ❌ Fail | Missing `total_scheduled`, `miss_rate_pct`, `last_missed_date`. |
| 14 | No hallucinations | ✅ Pass | No invented schema or dialect features. |
| 15 | Edge cases | ⚠️ Partial | Ties are broken by name, which is deterministic but meaningless for prioritization. |

**What works correctly:** schema use, the join, status filtering, date anchoring, grouping, and count arithmetic. The query is *technically* correct for the question it was asked.

**What is unclear or incorrect:** it answers "who missed the most appointments," not "who is least engaged." A manager reading this output would prioritize Brandon and Maya and could reasonably overlook Quinn, the only patient who attended **none** of their appointments. Nothing in the output reveals this, because the information needed to see it (total scheduled) is filtered out before aggregation.

## Task 4: Issues and Risks

| # | Issue | Type | Severity | Why it matters / impact on decisions |
|---|---|---|---|---|
| 1 | **No denominator.** The `WHERE` clause filters to missed statuses *before* grouping, so completed appointments are discarded and `total_scheduled` can't be computed. | Missing condition / structural logic | 🔴 High | Missed counts can't be interpreted. 2 missed out of 2 and 2 missed out of 10 look identical, so the team can't distinguish disengaged patients from busy but mostly reliable ones. |
| 2 | **Misleading ranking.** Ties on raw count are broken alphabetically. | Weak business logic | 🔴 High | The order *looks* like a priority list but partly reflects name spelling. Quinn Lewis (100% missed) ranks below Maya Stewart (67%). Outreach working down the list would reach the highest-risk patient late. |
| 3 | **Cancellations treated as patient risk.** No-shows and cancellations are summed into one total, but the data doesn't record who canceled. | Untested assumption | 🟠 Medium | A clinic- or provider-initiated cancellation isn't a patient behavior problem. Combining them can overstate a patient's risk. Brandon Carter's total of 3 includes 1 cancellation. |
| 4 | **No recency.** The output doesn't show when the last missed appointment happened. | Missing output | 🟠 Medium | Staff can't tell whether the problem is current or from three months ago, which affects how urgently to reach out. |
| 5 | **Validation only confirmed the prompt, not the goal.** Last lab's checks proved the SQL did what the prompt asked, but none of them tested whether the prompt asked the right question. | Workflow risk | 🟠 Medium | A query can pass every technical check and still lead to the wrong decision. That's exactly what happened here. |
| 6 | **Small samples per patient.** Each at-risk patient has only 2–4 appointments in the window. | Data limitation | 🟡 Low | Rates built on tiny denominators are volatile (one appointment shifts the rate by 25–50 points). The output should show `total_scheduled` next to the rate so readers can judge it. |

**Most important risk:** Issues 1 and 2 together. The output is clean, correct-looking, and passes its checks, but it ranks patients in a way that **hides the least-engaged patient**. That is the hardest kind of error to catch, because nothing fails.

**How the revision must address these:**
- Move the status filter from `WHERE` into conditional aggregation, so all appointments in the window are counted (fixes 1).
- Add `miss_rate_pct` using decimal division, and sort by it first (fixes 2).
- Keep `no_show_count` and `canceled_count` as separate columns, and state the limitation (mitigates 3).
- Add `last_missed_date` (fixes 4).
- Validate against the *business question* and an independent recomputation, not just the prompt (fixes 5).
- Show `total_scheduled` alongside the rate (mitigates 6).

## Task 5: Multi-Step Prompt Chain

All four prompts are submitted **in order, in one fresh AI chat**, so each step builds on the previous response.

| Prompt | Purpose | Addresses issues |
|---|---|---|
| 1. Clarify | The AI restates the task and flags ambiguities **before** writing SQL | 3, 5 |
| 2. Generate | The AI writes SQL using the clarified requirements | 1, 2, 4, 6 |
| 3. Review | The AI audits its own SQL against the validation checklist | 5 |
| 4. Refine | The AI fixes anything the review found and returns final SQL | All |

### Prompt 1: Clarify the task
```text
I'm a healthcare data analyst refining a SQLite query. Do NOT write SQL yet.

Context: This existing query lists patients with 2+ missed appointments in a recent window:

SELECT p.patient_id, p.patient_name, p.insurance_type,
  SUM(CASE WHEN a.appointment_status = 'no_show' THEN 1 ELSE 0 END) AS no_show_count,
  SUM(CASE WHEN a.appointment_status = 'canceled' THEN 1 ELSE 0 END) AS canceled_count,
  COUNT(*) AS total_missed
FROM patients p JOIN appointments a ON p.patient_id = a.patient_id
WHERE a.appointment_date >= (SELECT recent_3_month_start FROM reporting_context)
  AND a.appointment_date <= (SELECT reporting_date FROM reporting_context)
  AND a.appointment_status IN ('no_show', 'canceled')
GROUP BY p.patient_id, p.patient_name, p.insurance_type
HAVING COUNT(*) >= 2
ORDER BY total_missed DESC, p.patient_name ASC;

Problem: It ranks by raw missed count with an alphabetical tie-break. A patient who missed
2 of 2 appointments ranks below one who missed 2 of 3. The business goal is to prioritize
outreach to the LEAST ENGAGED patients, so we need each patient's miss rate.

Schema:
- patients(patient_id, patient_name, date_of_birth, city, insurance_type)
- appointments(appointment_id, patient_id, provider_id, clinic_id, appointment_date,
  appointment_status, appointment_type); appointment_date is TEXT 'YYYY-MM-DD';
  appointment_status values are exactly 'completed', 'no_show', 'canceled'
- reporting_context: one row with recent_3_month_start and reporting_date (does not join)

Please: (1) restate the business task in your own words, (2) list what the new query must
calculate, and (3) list any ambiguities or risks I should resolve before you write it.
```

### Prompt 2: Generate the revised SQL
```text
Thanks. Use these decisions to write the revised SQLite query:

- Recent window: appointment_date between recent_3_month_start and reporting_date from
  reporting_context, inclusive. Do not use date('now').
- total_scheduled = ALL appointments in the window, any status (the denominator).
- Do NOT filter appointment_status in WHERE; count statuses with conditional aggregation
  so completed appointments stay in the denominator.
- no_show_count and canceled_count stay separate (we can't tell who canceled).
- total_missed = no_show_count + canceled_count. Keep patients with total_missed >= 2.
- miss_rate_pct = 100.0 * total_missed / total_scheduled, rounded to 1 decimal
  (avoid SQLite integer division).
- last_missed_date = most recent appointment_date that was a no_show or canceled.

Output columns, in order: patient_id, patient_name, insurance_type, total_scheduled,
no_show_count, canceled_count, total_missed, miss_rate_pct, last_missed_date.
Sort: miss_rate_pct DESC, total_missed DESC, patient_name ASC.
Return only the SQL.
```

### Prompt 3: Review the SQL
```text
Now act as a reviewer, not a generator. Audit the query you just wrote against this
checklist and report PASS/FAIL with a one-line reason for each:

1. Uses only real tables and columns from the schema I gave
2. Join is patients.patient_id = appointments.patient_id and cannot duplicate rows
3. Status values match exactly ('no_show', 'canceled')
4. Window is anchored to reporting_context and inclusive on both ends
5. total_scheduled includes all statuses in the same window as the missed counts
6. miss_rate_pct cannot suffer SQLite integer division
7. The >= 2 threshold applies to missed appointments, not scheduled ones
8. last_missed_date only considers missed appointments
9. Output columns and sort order match my specification exactly
10. Valid SQLite syntax (no dialect-specific functions)

Do not rewrite the query yet. List any issues found.
```

### Prompt 4: Refine
```text
Fix every issue your review identified (if none, say so and keep the query unchanged).
Return the final SQLite query, followed by one sentence per change explaining what you
fixed and why.
```

## Task 6: Prompt Chain Results and Revised SQL

**AI assistant used:** Grok (xAI), with all four prompts in one fresh chat.

**Response 1 (Clarify), summarized:** Grok restated the goal correctly (rank by miss rate, not raw count) and raised 8 ambiguities before writing any SQL. Among them: whether the denominator includes completed appointments, whether to keep the 2-missed threshold, how to handle 1-of-1 patients, tie-breakers, and rate formatting. **Prompt 2 answered each of these explicitly**, which is why the SQL needed no correction.

**Response 2 (Generate): revised SQL, unedited:**
```sql
SELECT
  p.patient_id,
  p.patient_name,
  p.insurance_type,
  COUNT(*) AS total_scheduled,
  SUM(CASE WHEN a.appointment_status = 'no_show' THEN 1 ELSE 0 END) AS no_show_count,
  SUM(CASE WHEN a.appointment_status = 'canceled' THEN 1 ELSE 0 END) AS canceled_count,
  SUM(CASE WHEN a.appointment_status IN ('no_show', 'canceled') THEN 1 ELSE 0 END) AS total_missed,
  ROUND(100.0 * SUM(CASE WHEN a.appointment_status IN ('no_show', 'canceled') THEN 1 ELSE 0 END) / COUNT(*), 1) AS miss_rate_pct,
  MAX(CASE WHEN a.appointment_status IN ('no_show', 'canceled') THEN a.appointment_date END) AS last_missed_date
FROM patients p
JOIN appointments a ON p.patient_id = a.patient_id
WHERE a.appointment_date >= (SELECT recent_3_month_start FROM reporting_context)
  AND a.appointment_date <= (SELECT reporting_date FROM reporting_context)
GROUP BY p.patient_id, p.patient_name, p.insurance_type
HAVING SUM(CASE WHEN a.appointment_status IN ('no_show', 'canceled') THEN 1 ELSE 0 END) >= 2
ORDER BY miss_rate_pct DESC, total_missed DESC, p.patient_name ASC;
```

**Response 3 (Review):** Grok marked all 10 checklist items PASS with a one-line reason for each, and reported no issues.

**Response 4 (Refine):** No issues were found, so the query was returned unchanged.

**Source of improvement:** every change came from **prompt refinement**. No manual SQL edits were made. The key structural fix (moving the status filter from `WHERE` into conditional aggregation so completed appointments remain in the denominator) was specified in Prompt 2.

**Caution:** the AI reviewing its own SQL is useful but not sufficient, since a model can confirm its own mistakes. The results are validated independently in Task 7.

In [14]:
# Task 7: validate the revised SQL independently of the AI's self-review
revised_sql = """
SELECT
  p.patient_id,
  p.patient_name,
  p.insurance_type,
  COUNT(*) AS total_scheduled,
  SUM(CASE WHEN a.appointment_status = 'no_show' THEN 1 ELSE 0 END) AS no_show_count,
  SUM(CASE WHEN a.appointment_status = 'canceled' THEN 1 ELSE 0 END) AS canceled_count,
  SUM(CASE WHEN a.appointment_status IN ('no_show', 'canceled') THEN 1 ELSE 0 END) AS total_missed,
  ROUND(100.0 * SUM(CASE WHEN a.appointment_status IN ('no_show', 'canceled') THEN 1 ELSE 0 END) / COUNT(*), 1) AS miss_rate_pct,
  MAX(CASE WHEN a.appointment_status IN ('no_show', 'canceled') THEN a.appointment_date END) AS last_missed_date
FROM patients p
JOIN appointments a ON p.patient_id = a.patient_id
WHERE a.appointment_date >= (SELECT recent_3_month_start FROM reporting_context)
  AND a.appointment_date <= (SELECT reporting_date FROM reporting_context)
GROUP BY p.patient_id, p.patient_name, p.insurance_type
HAVING SUM(CASE WHEN a.appointment_status IN ('no_show', 'canceled') THEN 1 ELSE 0 END) >= 2
ORDER BY miss_rate_pct DESC, total_missed DESC, p.patient_name ASC;
"""
revised_result = pd.read_sql(revised_sql, conn)

# Independent recomputation in pandas from raw rows
ctx = pd.read_sql("SELECT recent_3_month_start, reporting_date FROM reporting_context;", conn).iloc[0]
appts = pd.read_sql("SELECT patient_id, appointment_date, appointment_status FROM appointments;", conn)
pts = pd.read_sql("SELECT patient_id, patient_name, insurance_type FROM patients;", conn)

window = appts[appts['appointment_date'].between(ctx['recent_3_month_start'], ctx['reporting_date'])].copy()
window['is_no_show'] = window['appointment_status'].eq('no_show').astype(int)
window['is_canceled'] = window['appointment_status'].eq('canceled').astype(int)
window['missed_date'] = window['appointment_date'].where(window['is_no_show'] + window['is_canceled'] > 0)

expected = (window.groupby('patient_id')
            .agg(total_scheduled=('appointment_status', 'size'),
                 no_show_count=('is_no_show', 'sum'),
                 canceled_count=('is_canceled', 'sum'),
                 last_missed_date=('missed_date', 'max'))
            .reset_index())
expected['total_missed'] = expected['no_show_count'] + expected['canceled_count']
expected = expected[expected['total_missed'] >= 2]
expected['miss_rate_pct'] = (100 * expected['total_missed'] / expected['total_scheduled']).round(1)
expected = (expected.merge(pts, on='patient_id')[revised_result.columns]
            .sort_values(['miss_rate_pct', 'total_missed', 'patient_name'],
                         ascending=[False, False, True])
            .reset_index(drop=True))

# Checklist assertions (item numbers from Task 2)
checks = {
    "#7  One row per patient": revised_result['patient_id'].is_unique,
    "#8  no_show + canceled = total_missed": (revised_result['no_show_count'] + revised_result['canceled_count']
                                              == revised_result['total_missed']).all(),
    "#9  total_missed <= total_scheduled": (revised_result['total_missed'] <= revised_result['total_scheduled']).all(),
    "#10 Decimal miss rate (no integer division)": (revised_result['miss_rate_pct'] > 0).all()
                                                   and revised_result['miss_rate_pct'].dtype == float,
    "#11 Threshold total_missed >= 2": (revised_result['total_missed'] >= 2).all(),
    "#12 Sorted by miss rate first": revised_result['miss_rate_pct'].is_monotonic_decreasing,
    "#13 All planned columns in order": list(revised_result.columns) == planned_columns,
    "#16 Matches independent pandas result": revised_result.equals(expected),
}
for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {name}")

revised_result

PASS  #7  One row per patient
PASS  #8  no_show + canceled = total_missed
PASS  #9  total_missed <= total_scheduled
PASS  #10 Decimal miss rate (no integer division)
PASS  #11 Threshold total_missed >= 2
PASS  #12 Sorted by miss rate first
PASS  #13 All planned columns in order
PASS  #16 Matches independent pandas result


,patient_id,patient_name,insurance_type,total_scheduled,no_show_count,canceled_count,total_missed,miss_rate_pct,last_missed_date
0,17,Quinn Lewis,Medicaid,2,2,0,2,100.0,2025-04-25
1,28,Brandon Carter,HMO,4,2,1,3,75.0,2025-06-05
2,39,Maya Stewart,Medicaid,3,2,0,2,66.7,2025-04-28
3,46,Thomas Bell,Medicare,3,2,0,2,66.7,2025-05-17


## Task 7: Validation Report: Before vs. After

### Checklist comparison

| # | Check | Original SQL | Revised SQL |
|---|---|---|---|
| 1–6 | Schema, join, status values, anchored window | ✅ | ✅ |
| 7–8 | One row per patient; counts reconcile | ✅ | ✅ |
| 9 | Denominator, same window, all statuses | ❌ Not possible (`WHERE` removed completed rows) | ✅ `total_scheduled` |
| 10 | Decimal miss rate | ❌ Absent | ✅ `100.0 *` prevents integer division |
| 11 | Threshold applied to missed appointments | ✅ | ✅ |
| 12 | Ranking reflects engagement | ❌ Count, then alphabetical | ✅ Miss rate first |
| 13 | All planned columns | ❌ 3 missing | ✅ All 9, in order |
| 16 | Independent verification | — | ✅ Identical to the pandas recomputation |

### Output comparison

| Patient | Missed / Scheduled | Miss rate | Last missed | Original rank | Revised rank |
|---|---|---|---|---|---|
| Quinn Lewis | 2 / 2 | 100.0% | 2025-04-25 | 3rd | **1st** |
| Brandon Carter | 3 / 4 | 75.0% | 2025-06-05 | 1st | 2nd |
| Maya Stewart | 2 / 3 | 66.7% | 2025-04-28 | 2nd | 3rd |
| Thomas Bell | 2 / 3 | 66.7% | 2025-05-17 | 4th | 4th |

### Evidence of improvement
- **The highest-risk patient is now first.** Quinn Lewis attended **none** of her scheduled appointments, and the revised query ranks her first instead of third.
- **New context for decisions.** `last_missed_date` together with `total_scheduled` shows that Quinn has had no appointments at all since April. She is not only missing visits but has stopped booking them, which may be the strongest outreach signal in the data. Brandon has the most recent misses, so his problem is the most current.
- **Independently verified.** The SQL result matches a separate pandas calculation from raw rows exactly. This confirms the AI's self-review with evidence instead of relying on it.
- **Same patient set.** Both queries return the same 4 patients. The revision changes the *interpretation and prioritization*, not who qualifies, which is consistent with keeping the 2-missed threshold.

### Remaining limitations
- **Cancellation ownership is unknown.** Brandon's total includes 1 cancellation that may have been clinic-initiated. Without that, his rate would be 2 of 4 (50%). The separate columns let staff judge this, but the data can't resolve it.
- **Small denominators.** With 2–4 appointments per patient, one appointment shifts the rate by 25–50 points. The rate should be read together with `total_scheduled`.
- **Rate vs. recency trade-off.** The sort uses miss rate only. A team that cares more about current behavior might rank Brandon first. That's a business choice for the manager, not a SQL error.
- **Single-miss patients excluded by design.** Eight patients missed their only appointment (1 of 1). They're excluded because a single event is too little evidence, but they could be a lighter-touch reminder list.

### Does the SQL meet the business goal?
**Yes.** It runs in SQLite, uses only valid schema, passes all 16 checklist items, matches an independent recomputation, and ranks patients by actual engagement, with enough context (denominator, recency, status split) for staff to make an informed outreach decision.

## Task 8: Reflection

**What was the most important issue I found?**
The original query was *correct but misleading*. It passed every technical check in the previous lab, yet it ranked the least-engaged patient third, partly because of alphabetical order. The root cause was structural: filtering statuses in the `WHERE` clause threw away completed appointments, so no denominator was possible. This issue was invisible from the output alone and only appeared when I compared each patient's misses with their total scheduled appointments.

**How did the prompt chain improve the SQL?**
Splitting the work into clarify → generate → review → refine gave each step a single job. The clarify step was the most valuable: before writing any SQL, the AI raised eight ambiguities (denominator, threshold, tie-breakers, 1-of-1 patients), and answering them explicitly in Prompt 2 produced correct SQL on the first attempt. In the previous lab, a single prompt led to guessed status values and the wrong dialect. Here, resolving ambiguity up front meant no corrections were needed.

**How did validation change my thinking?**
I used to treat "the query runs and the numbers add up" as success. This lab showed that checks can confirm a query does what the *prompt* asked without confirming that the prompt asked the right *business* question. I also saw that an AI reviewing its own work isn't proof. Grok passed itself on all 10 items, and that was only trustworthy after an independent pandas recomputation matched exactly. Validation has to test the business goal and use a separate method, not just re-read the SQL.

**How does this apply to real-world data workflows?**
In healthcare, an outreach list ordered by the wrong metric directly changes which patients get help first. Reliable AI-assisted SQL needs: (1) a written definition of "correct" before generating anything, (2) prompts that state exact values, dialect, and output, (3) a checklist covering business logic as well as syntax, (4) independent verification, and (5) clearly stated limitations (here, unknown cancellation ownership and small denominators), so decision-makers know how far to trust the results.

In [15]:
conn.close()
print("Database connection closed.")

Database connection closed.
